AQUEST CODI ACONSEGUEIX BAIXAR LES DADES FUNDAMENTALS DELS ANYS 2014-2025

In [ ]:
import requests, pandas as pd, re, time
from bs4 import BeautifulSoup
from datetime import datetime

# -------- CONFIG --------
USER_AGENT = "Jon jonmcgowan15@gmail.com"
HEADERS = {"User-Agent": USER_AGENT}
CIK = "0000320193"
FORMS = {"10-K", "10-Q"}
START = "2001-09-01"
END   = "2025-08-14"
PAUSE = 0.2

LABELS = {
    "revenue": [r"^revenue", r"net sales", r"sales,? net"],
    "net_income": [r"net income", r"net (loss|income)", r"profit attributable.*?shareholders"],
    "assets": [r"total assets"],
    "liabilities": [r"total liabilities"],
    "equity": [r"(total )?(share|stock)holders[’']? equity", r"total (members’|partners’) equity", r"total equity"]
}

# ----- UTIL -----
def parse_number(s: str, multiplier=1):
    if s is None: return None
    neg = bool(re.search(r"\(\s*[\d,\.]+\s*\)", s))
    t = re.sub(r"[^\d\.\-]", "", s)
    if t in {"", "-", "."}: return None
    try:
        val = float(t) * multiplier
        return -val if neg and val > 0 else val
    except:
        return None

def in_range(date_str):
    d = datetime.strptime(date_str, "%Y-%m-%d").date()
    return datetime.strptime(START, "%Y-%m-%d").date() <= d <= datetime.strptime(END, "%Y-%m-%d").date()

def pick_primary_doc_url(cik_int, accession_no, primary_doc):
    return f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{accession_no.replace('-', '')}/{primary_doc}"

def detect_multiplier(soup):
    text = soup.get_text(" ", strip=True).lower()
    if "in thousands" in text: return 1_000
    if "in millions" in text: return 1_000_000
    return 1

def extract_period_end(html_text: str):
    """
    Try to find the period end date inside the filing HTML text.
    Looks for common SEC phrases like 'For the three months ended June 30, 2009'
    """
    text = " ".join(html_text.split())
    # Common SEC phrasing
    m = re.search(r"ended\s+(?:on\s+)?([A-Za-z]+\s+\d{1,2},\s+\d{4})", text, re.I)
    if not m:
        m = re.search(r"as of\s+([A-Za-z]+\s+\d{1,2},\s+\d{4})", text, re.I)
    if m:
        try:
            return datetime.strptime(m.group(1), "%B %d, %Y").strftime("%Y-%m-%d")
        except:
            return None
    return None

def extract_metrics_from_html(html):
    soup = BeautifulSoup(html, "html.parser")
    multiplier = detect_multiplier(soup)
    metrics = {k: None for k in LABELS}

    def get_value_from_row(label_keywords):
        label_pattern = re.compile("|".join(label_keywords), re.I)
        for tr in soup.find_all("tr"):
            cells = tr.find_all(["td", "th"])
            if cells and any(label_pattern.search(c.get_text(strip=True)) for c in cells):
                for cell in cells[1:]:
                    text = cell.get_text(strip=True).replace(",", "")
                    if re.match(r"^-?\d+(\.\d+)?$", text):
                        return float(text) * multiplier
        return None

    metrics["revenue"]     = get_value_from_row(LABELS["revenue"])
    metrics["net_income"]  = get_value_from_row(LABELS["net_income"])
    metrics["assets"]      = get_value_from_row(LABELS["assets"])
    metrics["liabilities"] = get_value_from_row(LABELS["liabilities"])
    metrics["equity"]      = get_value_from_row(LABELS["equity"])

    return metrics

# ----- GET FILINGS -----
def get_all_filings(cik):
    sub_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    j = requests.get(sub_url, headers=HEADERS, timeout=30).json()

    def pack_from_recent(recent_dict):
        return pd.DataFrame({
            "accession": recent_dict["accessionNumber"],
            "form": recent_dict["form"],
            "filed": recent_dict["filingDate"],
            "primary": recent_dict["primaryDocument"]
        })

    recent_df = pack_from_recent(j["filings"]["recent"])
    older_parts = []
    for f in j.get("filings", {}).get("files", []):
        url = f"https://data.sec.gov/submissions/{f['name']}"
        y = requests.get(url, headers=HEADERS, timeout=30).json()
        if "filings" in y and "recent" in y["filings"]:
            older_parts.append(pack_from_recent(y["filings"]["recent"]))
        time.sleep(PAUSE)

    all_filings = pd.concat([recent_df] + older_parts, ignore_index=True)
    all_filings["filed"] = pd.to_datetime(all_filings["filed"]).dt.strftime("%Y-%m-%d")
    return all_filings.sort_values("filed")

# ----- MAIN -----
def main():
    cik_int = int(CIK)
    filings = get_all_filings(CIK)
    filings = filings[filings["form"].isin(FORMS)]
    filings = filings[filings["filed"].apply(in_range)]

    out_rows = []
    for _, r in filings.iterrows():
        url = pick_primary_doc_url(cik_int, r["accession"], r["primary"])
        try:
            html = requests.get(url, headers=HEADERS, timeout=30).text
            m = extract_metrics_from_html(html)
            p_end = extract_period_end(html)

            out_rows.append({
                "filing_date": r["filed"],     # when filed
                "period_end": p_end,           # actual period covered
                "form": r["form"],
                "revenue": m["revenue"],
                "net_income": m["net_income"],
                "total_assets": m["assets"],
                "total_liabilities": m["liabilities"],
                "shareholders_equity": m["equity"]
            })
            time.sleep(PAUSE)
        except:
            out_rows.append({
                "filing_date": r["filed"],
                "period_end": None,
                "form": r["form"],
                "revenue": None,
                "net_income": None,
                "total_assets": None,
                "total_liabilities": None,
                "shareholders_equity": None
            })
            time.sleep(PAUSE)

    df = pd.DataFrame(out_rows).sort_values(["filing_date","form"])
    df.to_csv("apple_financials_2014_2025.csv", index=False)
    print(f"Saved {len(df)} filings to financials_with_period_end.csv")

if __name__ == "__main__":
    main()


Saved 44 filings to financials_with_period_end.csv
